# Name: Pros Loung
# Course: AAI-540 Machine Learning Operations (MLOps)
# Assignment 2.1: Data Lake - Exercise
# GitHub: https://github.com/ploung/aai-540-homework.git

In [12]:
# Grab path to the dataset.csv

from pathlib import Path

current_directory = Path.cwd()

print("Current directory:")
print(current_directory)

print("\nCSV files found:")
csv_files = list(current_directory.rglob("*.csv"))

if not csv_files:
    print("No CSV files found.")
else:
    for file_path in csv_files:
        print(file_path.resolve())

Current directory:
/home/sagemaker-user/aai-540-homework/homework-2-1/data

CSV files found:
/home/sagemaker-user/aai-540-homework/homework-2-1/data/dataset.csv
/home/sagemaker-user/aai-540-homework/homework-2-1/data/.ipynb_checkpoints/dataset-checkpoint.csv


# Set up S3 Bucket for homework-2-1 dataset.csv

In [13]:
from io import BytesIO
from pathlib import Path

import boto3
import pandas as pd


bucket = "sagemaker-us-east-1-944202758041"
region = "us-east-1"

# Path the homework-2-1 dataset.csv
local_file = Path("/home/sagemaker-user/aai-540-homework/homework-2-1/data/.ipynb_checkpoints/dataset-checkpoint.csv")

s3_key = "homework-dataset/dataset.csv"
s3_uri = f"s3://{bucket}/{s3_key}"


if not local_file.is_file():
    raise FileNotFoundError(
        f"File not found: {local_file.resolve()}"
    )

print("Local file:", local_file.resolve())
print("Local file size:", local_file.stat().st_size, "bytes")


# Establish connection to Sagekmaker
session = boto3.session.Session(region_name=region)
s3 = session.client("s3")


# Upload dataset.csv to S3 Bucket
s3.upload_file(
    Filename=str(local_file),
    Bucket=bucket,
    Key=s3_key,
)

print("Uploaded to:")
print(s3_uri)


response = s3.head_object(
    Bucket=bucket,
    Key=s3_key,
)

print("S3 object verified")
print("S3 object size:", response["ContentLength"], "bytes")


object_response = s3.get_object(
    Bucket=bucket,
    Key=s3_key,
)

df = pd.read_csv(
    BytesIO(object_response["Body"].read())
)

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

display(df.head())

Local file: /home/sagemaker-user/aai-540-homework/homework-2-1/data/.ipynb_checkpoints/dataset-checkpoint.csv
Local file size: 20118244 bytes
Uploaded to:
s3://sagemaker-us-east-1-944202758041/homework-dataset/dataset.csv
S3 object verified
S3 object size: 20118244 bytes
Dataset loaded successfully
Dataset shape: (114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [14]:
s3_private_path_csv = (
    "s3://sagemaker-us-east-1-944202758041/"
    "homework-dataset/dataset.csv"
)

%store s3_private_path_csv

print(s3_private_path_csv)

Stored 's3_private_path_csv' (str)
s3://sagemaker-us-east-1-944202758041/homework-dataset/dataset.csv


# Display SageMaker S3 Bucket

In [15]:
from IPython.display import HTML, display

display(
    HTML(
        '<b>Open <a target="_blank" href="https://s3.console.aws.amazon.com/s3/buckets/{}?region={}&tab=objects">'
        "your SageMaker S3 bucket</a></b>".format(
            bucket,
            region,
        )
    )
)

# Create Athena Database Schema

In [16]:
# Check path the dataset.csv

%store -r s3_private_path_csv
print(s3_private_path_csv)

s3://sagemaker-us-east-1-944202758041/homework-dataset/dataset.csv


In [17]:
# Set up Athena Database
database_name = "database_athena"

In [29]:
# Set S3 staging directory -- this is a temporary directory used for Athena queries
s3_staging_dir = "s3://{0}/athena/staging".format(bucket)

In [30]:
# Create connection to Athena database
# import Pyathena to interact with Athena query service

from pyathena import connect 

conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

In [31]:
# Create the databsse if not exist

statement = "CREATE DATABASE IF NOT EXISTS {}".format(database_name)
print(statement)

CREATE DATABASE IF NOT EXISTS database_athena


In [32]:
# Use pandas to connect and query the database
import pandas as pd

pd.read_sql(statement, conn)

/tmp/ipykernel_530/2880253876.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


""


## Verify The Database Has Been Created Successfully

In [33]:
statement = "SHOW DATABASES"

df_show = pd.read_sql(statement, conn)
df_show.head(5)

/tmp/ipykernel_530/3999478089.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_show = pd.read_sql(statement, conn)


,database_name
0,database_athena
1,default
2,dsoaws


In [35]:
if database_name in df_show.values:
    ingest_create_athena_db_passed = True

%store ingest_create_athena_db_passed

Stored 'ingest_create_athena_db_passed' (bool)


In [36]:
%store

Stored variables and their in-db values:
ingest_create_athena_db_passed                        -> True
ingest_create_athena_table_parquet_passed             -> True
ingest_create_athena_table_tsv_passed                 -> True
s3_private_path_csv                                   -> 's3://sagemaker-us-east-1-944202758041/homework-da
s3_private_path_tsv                                   -> 's3://sagemaker-us-east-1-944202758041/amazon-revi
s3_public_path_tsv                                    -> 's3://aai-540/amazon-reviews-pds/tsv'
setup_dependencies_passed                             -> True
setup_s3_bucket_passed                                -> True


# Register Dataset.csv in Athena Database

In [37]:
ingest_create_athena_table_homeworkkdataset_passed = False

In [38]:
%store -r ingest_create_athena_db_passed

try:
    ingest_create_athena_db_passed
except NameError:
    print("++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN ALL PREVIOUS NOTEBOOKS.  You did not create the Athena Database.")
    print("++++++++++++++++++++++++++++++++++++++++++++++")

print(ingest_create_athena_db_passed)

True


In [39]:
# Import pyathena

from pyathena import connect 

# Create Athena Table from dataset.csv

## Dataset Columns
- track_id	
- artists	
- album_name
- track_name	
- popularity	
- duration_ms	
- explicit	
- danceability	
- energy
- key	
- loudness	
- mode	
- speechiness	
- acousticness	
- instrumentalness	
- liveness	
- valence	
- tempo	
- time_signature	
- track_genre


In [40]:
# Set S3 staging directory -- this is a temporary directory used for Athena queries
s3_staging_dir = "s3://{0}/athena/staging".format(bucket)

In [52]:
# Set Athena parameters
database_name = "database_athena"
table_name_csv = "music_tracks"

In [53]:
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

# Drop incorrect table and recreate correct table

In [67]:
database_name = "database_athena"
table_name_csv = "music_tracks"

cursor = conn.cursor()

cursor.execute(
    f"DROP TABLE IF EXISTS {database_name}.{table_name_csv}"
)

print(f"Dropped table: {database_name}.{table_name_csv}")

Dropped table: database_athena.music_tracks


# SQL Statement to execute to create table

In [68]:
s3_private_path_csv = (
    "s3://sagemaker-us-east-1-944202758041/homework-dataset/"
)

statement = f"""
CREATE EXTERNAL TABLE {database_name}.{table_name_csv} (
    row_number int,
    track_id string,
    artists string,
    album_name string,
    track_name string,
    popularity int,
    duration_ms int,
    explicit boolean,
    danceability double,
    energy double,
    track_key int,
    loudness double,
    mode boolean,
    speechiness double,
    acousticness double,
    instrumentalness double,
    liveness double,
    valence double,
    tempo double,
    time_signature int,
    track_genre string
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    'separatorChar' = ',',
    'quoteChar' = '"',
    'escapeChar' = '\\\\'
)
STORED AS TEXTFILE
LOCATION '{s3_private_path_csv}'
TBLPROPERTIES (
    'skip.header.line.count' = '1',
    'use.null.for.invalid.data' = 'true'
)
"""

cursor.execute(statement)

print(f"Created table: {database_name}.{table_name_csv}")

Created table: database_athena.music_tracks


# Verify table is created successfully

In [70]:
query = f"""
SELECT
    track_name,
    artists,
    track_key,
    loudness,
    tempo,
    time_signature,
    track_genre
FROM {database_name}.{table_name_csv}
LIMIT 10
"""

cursor.execute(query)

columns = [column[0] for column in cursor.description]
df_preview = pd.DataFrame(
    cursor.fetchall(),
    columns=columns,
)

display(df_preview)

,track_name,artists,track_key,loudness,tempo,time_signature,track_genre
0,Comedy,Gen Hoshino,1,-6.746,87.917,4,acoustic
1,Ghost - Acoustic,Ben Woodward,1,-17.235,77.489,4,acoustic
2,To Begin Again,Ingrid Michaelson;ZAYN,0,-9.734,76.332,4,acoustic
3,Can't Help Falling In Love,Kina Grannis,0,-18.515,181.740,3,acoustic
4,Hold On,Chord Overstreet,2,-9.681,119.949,4,acoustic
5,Days I Will Remember,Tyrone Wells,6,-8.807,98.017,4,acoustic
6,Say Something,A Great Big World;Christina Aguilera,2,-8.822,141.284,3,acoustic
7,I'm Yours,Jason Mraz,11,-9.331,150.960,4,acoustic
8,Lucky,Jason Mraz;Colbie Caillat,0,-8.700,130.088,4,acoustic
9,Hunger,Ross Copperman,1,-6.770,78.899,4,acoustic


In [71]:
query = f"""
SELECT COUNT(*) AS row_count
FROM {database_name}.{table_name_csv}
"""

cursor.execute(query)

row_count = cursor.fetchone()[0]
print("Athena row count:", row_count)

Athena row count: 114000


# Question: Show "track_names" and "energy" of tracks with energy above 0.5.

In [76]:
statement = f"""
SELECT
    track_name,
    energy
FROM {database_name}.{table_name_csv}
WHERE energy > 0.5
ORDER BY energy DESC
LIMIT 20
"""

cursor = conn.cursor()
cursor.execute(statement)

rows = cursor.fetchall()
columns = [column[0] for column in cursor.description]

df_energy_above_half = pd.DataFrame(
    rows,
    columns=columns,
)

print(f"Rows returned: {len(df_energy_above_half)}")
display(df_energy_above_half)

Rows returned: 20


,track_name,energy
0,March Rain,1.0
1,Amigo Charly Brown,1.0
2,Licht am Fahrrad,1.0
3,Soggy Afternoon,1.0
4,Calming Sea Waves,1.0
5,Bass Be Louder - Edit,1.0
6,Da sprach der alte Häuptling der Indianer,1.0
7,Transilvanian Hunger - Studio,1.0
8,Do Lacre ao Lucro,1.0
9,Gentle Rain Sounds,1.0


# Execute query statements
1) List artist, track_name, and popularity for songs that have a popularity greater than or equal to 99
2) List artists with an average popularity of 92
3) List the Top 10 genres with the highest average energy
4) How many tracks is Bad Bunny on?
5) Show the top 10 genres in terms of popularity, sorted by their most popular track

## 1. List artist, track_name, and popularity for songs that have a popularity greater than or equal to 99

In [77]:
statement = f"""
SELECT
    artists,
    track_name,
    popularity
FROM {database_name}.{table_name_csv}
WHERE popularity >= 99
ORDER BY popularity DESC, artists, track_name
"""

cursor = conn.cursor()
cursor.execute(statement)

rows = cursor.fetchall()
columns = [column[0] for column in cursor.description]

df_popular_tracks = pd.DataFrame(
    rows,
    columns=columns,
)

print(f"Tracks returned: {len(df_popular_tracks)}")
display(df_popular_tracks)

Tracks returned: 3


,artists,track_name,popularity
0,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),100
1,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),100
2,Bizarrap;Quevedo,"Quevedo: Bzrp Music Sessions, Vol. 52",99


# 2. List artists with an average popularity of 92

In [78]:
statement = f"""
SELECT
    artists,
    ROUND(AVG(popularity), 2) AS average_popularity
FROM {database_name}.{table_name_csv}
GROUP BY artists
HAVING ROUND(AVG(popularity), 2) = 92.00
ORDER BY artists
"""

cursor = conn.cursor()
cursor.execute(statement)

rows = cursor.fetchall()
columns = [column[0] for column in cursor.description]

df_artists_avg_92 = pd.DataFrame(
    rows,
    columns=columns,
)

print(f"Artists returned: {len(df_artists_avg_92)}")
display(df_artists_avg_92)

Artists returned: 2


,artists,average_popularity
0,Harry Styles,92.0
1,Rema;Selena Gomez,92.0


## 3. List the Top 10 genres with the highest average energy

In [80]:
statement = f"""
SELECT
    track_genre,
    ROUND(AVG(energy), 4) AS average_energy
FROM {database_name}.{table_name_csv}
GROUP BY track_genre
ORDER BY average_energy DESC
LIMIT 10
"""

cursor = conn.cursor()
cursor.execute(statement)

rows = cursor.fetchall()
columns = [column[0] for column in cursor.description]

df_top_genres_energy = pd.DataFrame(
    rows,
    columns=columns,
)

display(df_top_genres_energy)

,track_genre,average_energy
0,death-metal,0.9315
1,grindcore,0.9242
2,metalcore,0.9145
3,happy,0.9110
4,hardstyle,0.9012
5,drum-and-bass,0.8766
6,black-metal,0.8749
7,heavy-metal,0.8740
8,party,0.8712
9,j-idol,0.8687


## 4. How many tracks is Bad Bunny on?

In [81]:
statement = f"""
SELECT
    COUNT(*) AS bad_bunny_track_count
FROM {database_name}.{table_name_csv}
WHERE LOWER(artists) LIKE '%bad bunny%'
"""

cursor = conn.cursor()
cursor.execute(statement)

bad_bunny_track_count = cursor.fetchone()[0]

print("Bad Bunny appears on:", bad_bunny_track_count, "tracks")

Bad Bunny appears on: 416 tracks


## 5. Show the top 10 genres in terms of popularity, sorted by their most popular track

In [82]:
statement = f"""
WITH ranked_tracks AS (
    SELECT
        track_genre,
        track_name,
        artists,
        popularity,
        ROW_NUMBER() OVER (
            PARTITION BY track_genre
            ORDER BY popularity DESC, track_name
        ) AS genre_rank
    FROM {database_name}.{table_name_csv}
)
SELECT
    track_genre,
    track_name,
    artists,
    popularity
FROM ranked_tracks
WHERE genre_rank = 1
ORDER BY popularity DESC, track_genre
LIMIT 10
"""

cursor = conn.cursor()
cursor.execute(statement)

rows = cursor.fetchall()
columns = [column[0] for column in cursor.description]

df_top_genres_popularity = pd.DataFrame(
    rows,
    columns=columns,
)

display(df_top_genres_popularity)

,track_genre,track_name,artists,popularity
0,dance,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,100
1,pop,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,100
2,hip-hop,"Quevedo: Bzrp Music Sessions, Vol. 52",Bizarrap;Quevedo,99
3,edm,I'm Good (Blue),David Guetta;Bebe Rexha,98
4,latin,La Bachata,Manuel Turizo,98
5,latino,La Bachata,Manuel Turizo,98
6,reggae,La Bachata,Manuel Turizo,98
7,reggaeton,La Bachata,Manuel Turizo,98
8,piano,I Ain't Worried,OneRepublic,96
9,rock,I Ain't Worried,OneRepublic,96


# Release Resource

In [84]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

# AI Assistance Disclosure
AI tools were used as a support resource during the development of this notebook. Assistance included helping with codes generation, debug notebook cells, search for information, and fixing grammatical errors.